# Building Predictive Features

A complete feature engineering toolkit for time series ML models using **polars-ts**.

This notebook walks through the full pipeline of transforming raw time series into
ML-ready training data. We cover:

| Category | Functions |
|---|---|
| **Lag & rolling** | `lag_features`, `rolling_features` |
| **Calendar & cyclical** | `calendar_features`, `fourier_features`, `time_embeddings` |
| **Encoding** | `target_encode`, `holiday_features`, `interaction_features` |
| **Transforms** | `log_transform` / `inverse_log_transform`, `boxcox_transform` / `inverse_boxcox_transform` |
| **Differencing** | `difference` / `undifference` |

**Datasets**: [Air Passengers](https://www.kaggle.com/datasets/rakannimer/air-passengers) (monthly, 1949-1960) and
[M4 Hourly](https://github.com/Mcompetitions/M4-methods) (sub-daily, multiple series).

**References**:
- Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, 3rd ed. ([otexts.com/fpp3](https://otexts.com/fpp3/))
- Loning et al., *Modern Time Series Forecasting with Python*, 2nd ed.
- Box, Jenkins, Reinsel & Ljung, *Time Series Analysis*, 5th ed.

In [ ]:
# Install polars-timeseries and hvplot if not already available
import importlib

if importlib.util.find_spec("polars_ts") is None:
    %pip install -q polars-timeseries[all]
if importlib.util.find_spec("hvplot") is None:
    %pip install -q hvplot

In [ ]:
try:
    import hvplot.polars  # noqa
except ImportError:
    pass  # hvplot is optional for visualization
import polars as pl

from polars_ts import (
    boxcox_transform,
    calendar_features,
    difference,
    fourier_features,
    holiday_features,
    interaction_features,
    inverse_boxcox_transform,
    inverse_log_transform,
    lag_features,
    log_transform,
    rolling_features,
    target_encode,
    time_embeddings,
    undifference,
)

In [ ]:
# Air passengers -- classic monthly series
air = pl.read_csv(
    "https://datasets-nixtla.s3.amazonaws.com/air-passengers.csv",
    try_parse_dates=True,
).with_columns(pl.lit("AP").alias("unique_id"))

# M4 hourly -- pick 3 series for variety
m4 = (
    pl.scan_parquet("https://datasets-nixtla.s3.amazonaws.com/m4-hourly.parquet")
    .filter(pl.col("unique_id").is_in(["H1", "H2", "H3"]))
    .collect()
)

print(f"Air passengers: {air.shape}")
print(f"M4 hourly (3 series): {m4.shape}")
air.head()

## 1. Lag Features

Lag features shift the target column backward in time so each row "sees" past values.
They are the single most important feature family for autoregressive ML models.

Choosing the right lags depends on domain knowledge:
- **lag 1** captures short-term momentum,
- **lag 7 / 14 / 28** capture weekly and monthly cycles.

In [ ]:
air_lagged = lag_features(air, lags=[1, 7, 14, 28])
print(f"Columns added: {[c for c in air_lagged.columns if 'lag' in c]}")
air_lagged.head(30)

## 2. Rolling Features

Rolling (window) statistics smooth out noise and expose local trends.
We compute rolling **mean** and **std** over windows of 7, 14, and 28 periods.

In [ ]:
air_rolled = rolling_features(air, windows=[7, 14, 28], aggs=["mean", "std"])
print(f"Rolling columns: {[c for c in air_rolled.columns if 'rolling' in c]}")

# Visualize the original series alongside rolling means
roll_cols = ["y"] + [c for c in air_rolled.columns if "rolling_mean" in c]
air_rolled.hvplot.line(
    "ds",
    roll_cols,
    title="Air Passengers: Original vs Rolling Means",
    height=350,
    width=750,
)

## 3. Calendar Features

Calendar features extract deterministic time components (day-of-week, month, quarter, etc.)
directly from the datetime column. These are invaluable for models that cannot
natively interpret dates, such as gradient-boosted trees.

In [ ]:
air_cal = calendar_features(
    air, features=["year", "month", "day_of_month", "day_of_week", "day_of_year", "week", "quarter"]
)
print(f"Calendar columns: {[c for c in air_cal.columns if c not in air.columns]}")
air_cal.head()

## 4. Fourier Features

Fourier terms encode cyclical patterns as pairs of sine/cosine columns.
Unlike one-hot calendar encoding, Fourier features are continuous and low-dimensional,
which is useful when the seasonal period is long (e.g., 365 days).

For the monthly Air Passengers data, `period=12` captures yearly seasonality.

In [ ]:
air_fourier = fourier_features(air, period=12.0, n_harmonics=3)
fourier_cols = [c for c in air_fourier.columns if "fourier" in c]
print(f"Fourier columns: {fourier_cols}")

# Plot the first harmonic pair to see the sinusoidal encoding
air_fourier.hvplot.line(
    "ds",
    [c for c in fourier_cols if "_1" in c],
    title="First Fourier Harmonic (period=12)",
    height=300,
    width=750,
)

## 5. Advanced Features

### Target Encoding
Replace a categorical column (e.g., month) with the smoothed mean of the target.

### Holiday Features
Add binary indicators and distance-to-holiday columns for a given country.

### Interaction Features
Create multiplicative interactions between selected pairs of existing features.

### Time Embeddings
Sine/cosine encoding of individual time components (hour, day-of-week, month),
useful as positional encodings in neural architectures.

In [ ]:
# Target encoding on the month extracted from calendar features
air_cal = calendar_features(
    air, features=["year", "month", "day_of_month", "day_of_week", "day_of_year", "week", "quarter"]
)
air_te = target_encode(air_cal, cat_col="month", smoothing=10.0)
print("Target-encoded columns:", [c for c in air_te.columns if "encoded" in c])

# Holiday features
air_hol = holiday_features(air, country="US", distance=True)
print("Holiday columns:", [c for c in air_hol.columns if "holiday" in c or "days_" in c])

# Interaction features -- first create base features, then interact them
air_base = lag_features(air, lags=[1])
air_base = rolling_features(air_base, windows=[7], aggs=["mean"])
air_inter = interaction_features(air_base, pairs=[("y_lag_1", "y_rolling_mean_7")])
print("Interaction columns:", [c for c in air_inter.columns if "interaction" in c or "_x_" in c])

# Time embeddings on M4 hourly (has sub-daily granularity)
m4_with_dt = m4.with_columns(
    (pl.datetime(1970, 1, 1).cast(pl.Datetime) + pl.duration(hours=pl.col("ds") - 1)).alias("ds_datetime")
)
m4_emb = time_embeddings(m4_with_dt, time_col="ds_datetime", components=["hour", "day_of_week", "month"])
emb_cols = [c for c in m4_emb.columns if "sin" in c or "cos" in c]
print(f"Time embedding columns: {emb_cols}")
m4_emb.filter(pl.col("unique_id") == "H1").head()

## 6. Log Transform (Round-Trip)

The log transform stabilizes variance in series with multiplicative seasonality.
`log_transform` applies `log1p` and preserves the original target in `y_original`.
`inverse_log_transform` reverses the operation exactly.

In [ ]:
air_log = log_transform(air)
air_restored = inverse_log_transform(air_log)

# Verify round-trip: the restored values should match the originals
max_error = (air["y"] - air_restored["y"]).abs().max()
print(f"Max round-trip error (log): {max_error}")

# Side-by-side visualization
original_plot = air.hvplot.line("ds", "y", title="Original", height=280, width=370)
log_plot = air_log.hvplot.line("ds", "y", title="Log-Transformed", height=280, width=370)
original_plot + log_plot

## 7. Box-Cox Transform (Round-Trip)

The Box-Cox family generalizes the log transform via a parameter `lambda`.
When `lam=0`, it reduces to the natural log; `lam=0.5` gives a square-root-like
transform; `lam=1` is the identity (no change).

The transform stores `y_original` and `y_boxcox_lambda` so the inverse can
recover the original scale without any extra bookkeeping.

In [ ]:
air_bc = boxcox_transform(air, lam=0.5)
air_bc_restored = inverse_boxcox_transform(air_bc)

max_error_bc = (air["y"] - air_bc_restored["y"]).abs().max()
print(f"Max round-trip error (Box-Cox, lam=0.5): {max_error_bc}")

original_plot = air.hvplot.line("ds", "y", title="Original", height=280, width=370)
bc_plot = air_bc.hvplot.line("ds", "y", title="Box-Cox (lam=0.5)", height=280, width=370)
original_plot + bc_plot

## 8. Differencing and Undifferencing

Differencing removes trend and/or seasonality from a series, making it closer
to stationary -- a prerequisite for many classical and ML models.

- **First-order differencing** (`order=1, period=1`): removes linear trend.
- **Seasonal differencing** (`order=1, period=12`): removes yearly seasonality
  in monthly data.

`undifference` reconstructs the original levels using the stored initial values.

In [ ]:
# First-order differencing
air_d1 = difference(air, order=1, period=1)
air_d1_restored = undifference(air_d1, order=1, period=1)
# Join to align the original and restored series for error calculation
compared_df_d1 = air.join(air_d1_restored, on=["unique_id", "ds"], how="inner", suffix="_restored")
err_d1 = (compared_df_d1["y"] - compared_df_d1["y_restored"]).abs().max()
print(f"Round-trip error (diff order=1): {err_d1}")

# Seasonal differencing (period=12 for monthly data)
air_d12 = difference(air, order=1, period=12)
air_d12_restored = undifference(air_d12, order=1, period=12)
# Join to align the original and restored series for error calculation
compared_df_d12 = air.join(air_d12_restored, on=["unique_id", "ds"], how="inner", suffix="_restored")
err_d12 = (compared_df_d12["y"] - compared_df_d12["y_restored"]).abs().max()
print(f"Round-trip error (seasonal diff period=12): {err_d12}")

# Visualize all three variants
p_orig = air.hvplot.line("ds", "y", title="Original", height=250, width=370)
p_d1 = air_d1.hvplot.line("ds", "y", title="First Difference", height=250, width=370)
p_d12 = air_d12.hvplot.line("ds", "y", title="Seasonal Difference (period=12)", height=250, width=370)
(p_orig + p_d1 + p_d12).cols(2)

## 9. Putting It All Together

A realistic ML pipeline chains several of the transforms above into a single
feature-rich DataFrame. Below we combine lag, rolling, calendar, and Fourier
features, then apply a log transform to stabilize the target variance.

In [ ]:
# Start with a log-transformed copy for variance stabilization
df = log_transform(air)

# Add lag features
df = lag_features(df, lags=[1, 7, 14, 28])

# Add rolling statistics
df = rolling_features(df, windows=[7, 14, 28], aggs=["mean", "std"])

# Add calendar columns
df = calendar_features(df, features=["year", "month", "day_of_month", "day_of_week", "day_of_year", "week", "quarter"])

# Add Fourier terms for yearly cycle
df = fourier_features(df, period=12.0, n_harmonics=3)

print(f"Final shape: {df.shape}")
print(f"Feature columns ({len(df.columns) - 3} features + unique_id, ds, y):")
print(df.columns)

# Drop rows with nulls from lagging/rolling, ready for model training
df_train = df.drop_nulls()
print(f"\nTraining rows (after dropping nulls): {df_train.shape[0]}")
df_train.head()

## Summary

We covered the full **polars-ts** feature engineering toolkit:

1. **Lag features** create autoregressive inputs from past target values.
2. **Rolling features** expose local trends and volatility through windowed statistics.
3. **Calendar features** extract deterministic time components for tree-based models.
4. **Fourier features** provide continuous, low-dimensional cyclical encodings.
5. **Target encoding**, **holiday features**, **interaction features**, and **time embeddings**
   round out the advanced feature set.
6. **Log and Box-Cox transforms** stabilize variance; both support exact round-trips.
7. **Differencing** removes trend and seasonality; `undifference` restores the original levels.

All functions accept and return Polars DataFrames, compose naturally via chaining,
and respect the `unique_id` / `ds` / `y` convention so they work seamlessly with
the rest of the polars-ts ecosystem.

### Further Reading

- [Forecasting: Principles and Practice (FPP3)](https://otexts.com/fpp3/) -- Ch. 7 (Time series regression) and Ch. 9 (ARIMA / differencing)
- [Feature Engineering for Machine Learning (Zheng & Casari, O'Reilly)](https://www.oreilly.com/library/view/feature-engineering-for/9781491953235/)
- [polars-ts documentation](https://github.com/pola-rs/polars-ts)